In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, TimestampType

In [0]:
env = "dev"
schema = "bronze"
table = "covid"
table_name = f"{env}.{schema}.{table}"

In [0]:
display(dbutils.fs.ls("dbfs:/databricks-datasets/COVID/coronavirusdataset/"))

In [0]:
df = spark.read.csv("dbfs:/databricks-datasets/COVID/coronavirusdataset/PatientInfo.csv", sep = ",", header = True)
df.count()

In [0]:
display(df.limit(5))

In [0]:
schema_dlt = StructType(
    fields=[
        StructField("infection_case", StringType()),
        StructField("province", StringType()),
        StructField("country", StringType()),
        StructField("city", StringType()),
        StructField("contact_number", IntegerType()),
        StructField("confirmed_date", DateType()),
    ]
)

###Treino do codigo autoLoader, porém com ambiente de estudo, não é possivel carregar o codigo

- df_streaming_autoloader = (
- 
- spark.readStream
-     .format("cloudFiles")
-     .option("cloudFiles.format", "csv")
-     .option("cloudFiles.schemaLocation", "dbfs:/FileStore/tables/COVID/schema")
-     .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
-     .option("cloudFiles.inferColumnTypes", "true")
-     .option("header", "true")
-     .option("sep", ",")
-     .load("dbfs:/databricks-datasets/COVID/coronavirusdataset/")
- )


In [0]:
df_streaming_autoloader = (

spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", "dbfs:/FileStore/tables/COVID/coronavirusdataset/_schema")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("header", "true")
    .option("sep", ",")
    .load("dbfs:/databricks-datasets/COVID/coronavirusdataset/")
)

In [0]:
# (df.writeStream
#   .format("delta")
#   .option("checkpointLocation", "abfss://container@storageaccount.dfs.core.windows.net/checkpoints/minha_ingestao/")
#   .trigger(availableNow=True)
#   .table("catalogo.schema.minha_tabela")
# )

### Write delta table

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {env}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {env}.{schema}")

In [0]:
def write_delta_table(df, table_name):
  (df.write
   .mode("overwrite")
   .format("delta")
   .saveAsTable(table_name)
  )

In [0]:
write_delta_table(df, table_name)